# pt-vision: what has been validated, and how to see it yourself

This notebook walks through the validation we have for the sit-to-stand pipeline and lets you poke
at it. Everything runs locally; nothing is uploaded.

What is covered:

1. **A real phone clip** (the bundled sample) through the whole pipeline, compared with a hand-timed
   stopwatch value.
2. **Robustness to partial visibility**: the same clip with the head cut off, to see what the tracker
   does when the body is not fully in frame.
3. **UI-PRMD**: 200 motion-capture sit-to-stand episodes (public domain) through the same segmenter,
   checking that each is counted as exactly one rise.

What is **not** covered yet: joint-angle accuracy against motion capture (planned on OpenCap and
COMFI, which have synchronised video, markers and force plates) and walking (the gait protocol is
not in this release).

## Setup

From the repo root, once:

```sh
uv sync --extra app --extra datasets --group dev
uv run ptv models pull --mode balanced        # 157 MB, once
uv run ptv datasets pull uiprmd               # 0.4 GB, once
uv run --with jupyter jupyter lab notebooks/validation.ipynb
```

Run the cells top to bottom. The first analysis takes about 10 s on a laptop; UI-PRMD takes a
minute or two. Outputs land in `ptv_out/notebook/` (gitignored).

In [ ]:
%matplotlib inline
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
OUT = ROOT / "ptv_out" / "notebook"
OUT.mkdir(parents=True, exist_ok=True)
print("repo:", ROOT)

## 1. A real clip through the pipeline

The bundled sample is a five-times sit-to-stand filmed with a phone on a tripod, slightly off the
side axis (the quality check will say so). The person who recorded it timed it with a stopwatch:
**10.75 s** from the first seat-off to the fifth full stand. We run the exact code path
`ptv analyze` uses.

In [ ]:
from ptvision.pipeline import AnalyzeOptions, analyze
from ptvision.samples import sample
from ptvision.trials.models import Subject

smp = sample()  # the bundled clip and its known subject details
res = analyze(
    smp.path,
    AnalyzeOptions(
        protocol=smp.protocol,
        out_dir=OUT / "sample",
        overlay=False,
        subject=Subject(height_m=smp.height_m, age_years=smp.age_years, sex=smp.sex),
    ),
)
print("run folder:", res.run.path)
print(f"stopwatch: {smp.stopwatch_s} s")
print(
    "quality:",
    res.quality.status,
    [c.name for c in res.quality.failures],
    [c.name for c in res.quality.warnings],
)
pd.DataFrame([{"metric": m.label, "value": m.formatted(), "tier": m.tier} for m in res.metrics])

The primary metric is the total time, with the error the tool assigns to it (two frames at 30 fps).
The stopwatch said 10.75 s; the tool's value is in the table. The gap is larger than the stated
error, and the likely reason is where each one stops: the tool ends at the moment the pelvis
reaches standing height, a person with a stopwatch stops when the body looks upright and still,
which comes a little later. A stopwatch also carries about ±0.1 s of reaction time. That systematic
offset is exactly what the PT session (blind timing of every clip) will measure, so the tool can
report it rather than hide it.

Every number above traces back to events on the pelvis-height signal. The figure the report writes
shows them: rises in green, standing in blue, descents in orange, the timed interval dashed.

In [ ]:
display(Image(filename=str(res.run.path / "figures" / "sts_trajectory.png"), width=800))
display(Image(filename=str(res.run.path / "figures" / "angles.png"), width=800))

And here is what the skeleton looked like at the first seat-off, drawn with the same confidence
colouring the viewer uses (green confident, amber uncertain, red barely tracked).

In [ ]:
from ptvision.pose.overlay import draw_pose
from ptvision.pose.track import PoseTrack
from ptvision.video import FrameSource

trial = res.trial
track = PoseTrack.load(trial.pose_path())
capture = trial.read_capture()
video = trial.path / capture.cameras[0].normalized_file
first_rep = res.events.reps[0]
frame_idx = int(first_rep.seat_off)
with FrameSource(video) as src:
    img = src.read(frame_idx).copy()
draw_pose(
    img,
    track.coords[frame_idx],
    track.score[frame_idx],
    track.layout,
    track.person_ids,
    primary=res.primary_person,
    mode="confidence",
    hidden=track.spurious,
)
plt.figure(figsize=(5, 8))
plt.imshow(img[..., ::-1])
plt.axis("off")
plt.title(f"frame {frame_idx}: first seat-off")
plt.show()
print(
    "people tracked:",
    track.n_persons_real,
    "of",
    track.n_persons,
    "tracks (the rest were short false detections)",
)

## 2. What happens when the body is not fully visible

Phone clips in a clinic will not always frame the whole person. Here we crop the top of the sample
clip off (the head and shoulders disappear for the standing phases) and run pose tracking again. The
things to look at: how many frames still have the subject, whether a second identity appears, and
whether joints outside the frame are drawn as if they were seen.

In [ ]:
import subprocess

from ptvision.pipeline import pose_only

cut = OUT / "head_cut.mp4"
if not cut.exists():
    subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-hide_banner",
            "-loglevel",
            "error",
            "-i",
            str(smp.path),
            "-vf",
            "crop=iw:ih*0.55:0:ih*0.45",
            "-c:v",
            "libx264",
            "-crf",
            "23",
            "-pix_fmt",
            "yuv420p",
            "-an",
            str(cut),
        ],
        check=True,
    )
pr = pose_only(cut, AnalyzeOptions(out_dir=OUT / "head_cut", overlay=False))
tr = pr.track
present = ~np.isnan(tr.coords).all(axis=(2, 3))
primary_slot = int(np.flatnonzero(tr.person_ids == pr.primary_person)[0])
print(f"frames with the subject tracked: {int(present[:, primary_slot].sum())} of {tr.n_frames}")
print(f"identities: {tr.n_persons_real} real, {tr.n_persons} tracks")
conf = tr.score[:, primary_slot]
names = tr.layout.names
print("mean confidence per joint group:")
for label, idx in {
    "head": [0, 1, 2, 3, 4],
    "shoulders": [5, 6],
    "hips": [11, 12],
    "knees": [13, 14],
    "ankles": [15, 16],
}.items():
    print(f"  {label:<10} {float(np.nanmean(conf[:, idx])):.2f}")

In [ ]:
cap = pr.run.path.parent.parent
cvideo = next((cap / "video").glob("*.mp4"))
fig, axes = plt.subplots(1, 3, figsize=(15, 6))
with FrameSource(cvideo) as src:
    for ax, f in zip(axes, (60, 120, 200), strict=True):
        im = src.read(f).copy()
        draw_pose(
            im,
            tr.coords[f],
            tr.score[f],
            tr.layout,
            tr.person_ids,
            primary=pr.primary_person,
            mode="confidence",
            hidden=tr.spurious,
        )
        ax.imshow(im[..., ::-1])
        ax.axis("off")
        ax.set_title(f"frame {f}")
plt.suptitle("Head cut off: tracked joints stay, missing ones are not invented")
plt.show()

Joints that are outside the frame get confidence zero and are not drawn; the rest of the body is
tracked as before, and the person keeps one identity through the frames where the detector loses
them. Before this was fixed, the same clip tracked the subject in only 98 of 268 frames.

## 3. UI-PRMD: does the segmenter count one rise as one rise?

UI-PRMD (University of Idaho, public domain) has 10 healthy subjects performing 10 sit-to-stand
episodes each, correctly and then deliberately incorrectly, recorded with a 39-marker Vicon system
at 100 Hz. Each file is exactly one repetition. We project the markers onto a virtual side camera so
they look like our 2D keypoints, then run the same segmenter the video pipeline uses. Success is
simple: every episode should yield exactly one rise. There is no timing ground truth in this set.

In [ ]:
from ptvision.datasets.uiprmd import evaluate_sts

results, out_dir = evaluate_sts(fps=30)
summary = json.loads((out_dir / "summary.json").read_text())
df = pd.DataFrame([r.__dict__ for r in results])
print(json.dumps(summary, indent=2))
pd.crosstab(df["correct"], df["reps"])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for correct, label in ((True, "performed correctly"), (False, "performed incorrectly")):
    vals = df[(df.correct == correct) & (df.reps == 1)].rise_time_s.dropna()
    ax.hist(vals, bins=20, alpha=0.6, label=f"{label} (n={len(vals)})")
ax.set_xlabel("rise time, seat-off to stand (s)")
ax.set_ylabel("episodes")
ax.legend()
ax.set_title("UI-PRMD: rise duration per episode")
plt.show()

The "incorrect" episodes are the same people exaggerating faults (asymmetric loading, trunk lean), so
they are a useful check that the segmenter does not fall apart on unusual movement. Below, one
episode's projected pelvis height with the detected seat-off and stand, so you can see what the
segmenter keys on.

In [ ]:
from ptvision.clinical.segmenters.sts import StsParams, segment_sts
from ptvision.datasets.mocap_projection import project_trial
from ptvision.datasets.uiprmd import episodes, read_vicon_positions
from ptvision.kinematics.preprocess import preprocess

path, correct = episodes("m05")[0]
pt = project_trial(read_vicon_positions(path), fps=30)
ev = segment_sts(
    preprocess(pt.track).filtered, 30, StsParams(n_reps_expected=1, end_rule="fifth_stand")
)
t = np.arange(len(ev.height_norm)) / 30
plt.figure(figsize=(9, 3.5))
plt.plot(t, ev.height_norm, label="pelvis height (normalised)")
for r in ev.reps:
    plt.axvline(r.seat_off / 30, color="green", ls="--", label="seat-off")
    plt.axvline(r.stand_reached / 30, color="blue", ls="--", label="stand reached")
plt.xlabel("time (s)")
plt.legend()
plt.title(path.name)
plt.show()

## 4. What this does and does not tell us

- **Counting and segmentation** work on clean marker data (200 of 200) and on a real phone clip.
- **Timing** agrees with a stopwatch on one real clip to within the stopwatch's own uncertainty. One
  clip is an anecdote, not a validation; the recording session with the PT (about 10 volunteers, each
  twice) is what turns this into a number with an interval.
- **Joint angles** have no ground truth here. That comes next from OpenCap and COMFI, which pair
  phone video with motion capture and force plates; the pipeline will be run frozen on those sets
  and sagittal hip and knee RMSE reported per subject.
- **Partial visibility** is handled without inventing joints, but the quality checks still refuse
  clips where the subject is out of frame too often, which is the right call for a measurement tool.

To explore further: change `frame_idx` above, try your own clip in place of `smp.path`
(`uv run ptv app` does the same with a viewer), or open `res.run.path / "report.html"`.